In [2]:
import pandas as pd
import sqlite3

**SPOILER**

**Miranda Priestly pago a Jeremy Bowers para que lo haga**

In [3]:
connection = sqlite3.connect("data/sql-murder-mystery.db")

# Obtenemos un cursor que utilizaremos para hacer las queries
crsr = connection.cursor()

In [4]:
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    crsr.execute(query)

    # Almacena los datos de la query 
    ans = crsr.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in crsr.description]

    return pd.DataFrame(ans,columns=names)

In [5]:
res = crsr.execute("SELECT name FROM sqlite_master WHERE type='table'")
for name in res:
    print(name[0])

crime_scene_report
drivers_license
person
facebook_event_checkin
interview
get_fit_now_member
get_fit_now_check_in
income
solution


In [6]:
#miro asesinatos
query = '''SELECT * FROM crime_scene_report 
WHERE type = 'murder' 
AND city = 'SQL City'
AND date ='20180115';
'''

sql_query(query)

,date,type,description,city
0,20180115,murder,Security footage shows that there were 2 witne...,SQL City


In [7]:
query = '''SELECT *FROM crime_scene_report

WHERE type = 'murder' 
AND city = 'SQL City'
AND date ='20180115';
'''

df = sql_query(query)
df.iloc[0,2]

'Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave".'

In [155]:
#Localizamos testigo 1
query = '''SELECT * 
FROM person
where address_street_name like "Northwestern Dr" and address_number= (select max (address_number) from  person
where address_street_name like "Northwestern Dr");
'''

sql_query(query)

,id,name,license_id,address_number,address_street_name,ssn
0,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949


In [156]:
#Localizamos testigo 2
query = '''SELECT * 
FROM person

where name like"Annabel%" and 
address_street_name like "Franklin Ave";
'''

sql_query(query)

,id,name,license_id,address_number,address_street_name,ssn
0,16371,Annabel Miller,490173,103,Franklin Ave,318771143


In [157]:
#Sospechoso 2
query = '''SELECT * 
FROM interview

where person_id like "16371";
'''

df=sql_query(query)
df.iloc[0,1]

'I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th.'

In [18]:
#Sospechoso 2
query = '''SELECT * 
FROM interview

where person_id like "14887";
'''

sql_query(query).iloc[0,1]# dbeaver

'I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W".'

In [158]:
# miramos la descripcion del sospechoso segun el testigo 2
query = '''SELECT * 
FROM get_fit_now_member

where id like '%48Z%' and
membership_status like"gold";
'''

sql_query(query)

,id,person_id,name,membership_start_date,membership_status
0,48Z7A,28819,Joe Germuska,20160305,gold
1,48Z55,67318,Jeremy Bowers,20160101,gold


In [ ]:
query = '''SELECT * 
FROM person
where license_id like "423327"
;'''
sql_query(query)

#Es el coche de Jeremy Bowers


,id,name,license_id,address_number,address_street_name,ssn
0,67318,Jeremy Bowers,423327,530,"Washington Pl, Apt 3A",871539279


In [163]:
query = '''SELECT * 
FROM drivers_license
where plate_number like "%H42W%"
and gender like "male";
'''

sql_query(query)
#la matricula no coincide pero haberlo robado

,id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model
0,423327,30,70,brown,brown,male,0H42W2,Chevrolet,Spark LS
1,664760,21,71,black,black,male,4H42WR,Nissan,Altima


In [ ]:
query = '''SELECT * 
FROM get_fit_now_check_in
where membership_id like "48Z7A" or "48Z55"and
check_in_date like "20180109";
'''

sql_query(query)
# Joe Germuska  y Jeremy Bowers fueron  al gimnasio ese dia que un testigo los reconocio

,membership_id,check_in_date,check_in_time,check_out_time
0,X0643,20180109,957,1164
1,UK1F2,20180109,344,518
2,XTE42,20180109,486,1124
3,1AE2H,20180109,461,944
4,6LSTG,20180109,399,515
5,7MWHJ,20180109,273,885
6,GE5Q8,20180109,367,959
7,48Z7A,20180109,1600,1730
8,48Z55,20180109,1530,1700
9,90081,20180109,1600,1700


In [ ]:
query = '''SELECT * 
FROM get_fit_now_check_in
where membership_id like "48Z7A" or "48Z55"and
check_in_date like "20180115";
'''

sql_query(query)
# Pero John Gremusca no al gimnasio el dia del asesinato lo no le daria cuartada

,membership_id,check_in_date,check_in_time,check_out_time
0,D2KY6,20180115,746,836
1,344VM,20180115,1087,1195
2,3BRSC,20180115,354,825
3,HM6U8,20180115,525,800
4,48Z7A,20180109,1600,1730


In [80]:

query = '''SELECT * 
FROM interview
where person_id like "%67318%";
'''
df=sql_query(query)

df.iloc[0,1]
# es el asesino pero quien le contrato?

'I was hired by a woman with a lot of money. I don\'t know her name but I know she\'s around 5\'5" (65") or 5\'7" (67"). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017.\n'

In [ ]:
query = '''SELECT * 
FROM drivers_license
where
car_make like"Tesla" and car_model like "Model S"
and gender like "female";
'''

sql_query(query)
#tres sospechosas 202298, 291182, 918773

,id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model
0,202298,68,66,green,red,female,500123,Tesla,Model S
1,291182,65,66,blue,red,female,08CM64,Tesla,Model S
2,682231,51,80,green,grey,female,3Z0M62,Tesla,Model S
3,918773,48,65,black,red,female,917UU3,Tesla,Model S


In [138]:
query = '''SELECT * 
FROM person
where license_id like "%202298%"; '''
sql_query(query)

,id,name,license_id,address_number,address_street_name,ssn
0,99716,Miranda Priestly,202298,1883,Golden Ave,987756388


In [ ]:
#miro la sospechosa 1
query = '''SELECT * 
FROM facebook_event_checkin
where event_name like "SQL Symphony Concert" and 
date like "%201712%"
and person_id  like "99716";
'''

sql_query(query)
#Es la unica de las tres que cumple con lo sabe el sicariovamos a interrogarla

,person_id,event_id,event_name,date
0,99716,1143,SQL Symphony Concert,20171206
1,99716,1143,SQL Symphony Concert,20171212
2,99716,1143,SQL Symphony Concert,20171229


In [146]:
query = '''SELECT * 
FROM person
where license_id like "%682231%"; '''
sql_query(query)

,id,name,license_id,address_number,address_street_name,ssn
0,67673,Alyssa Faggett,682231,3838,Mc Guffie Dr,666309770


In [145]:
#miro la sospechosa 1
query = '''SELECT * 
FROM facebook_event_checkin
where event_name like "SQL Symphony Concert" and 
date like "%201712%"
and person_id  like "67673";
'''

sql_query(query)

,person_id,event_id,event_name,date


In [139]:
query = '''SELECT * 
FROM person
where license_id like "%918773%"; '''
sql_query(query)

,id,name,license_id,address_number,address_street_name,ssn
0,78881,Red Korb,918773,107,Camerata Dr,961388910


In [149]:
#miro la sospechosa 1
query = '''SELECT * 
FROM facebook_event_checkin
where event_name like "SQL Symphony Concert" and 
date like "%201712%"
and person_id  like "78881";
'''

sql_query(query)

,person_id,event_id,event_name,date


In [ ]:
query = '''SELECT * 
FROM interview
where person_id like "99716";
'''
sql_query(query)
#no se puede pero si metes en la pagian web Miranda Priestly confirma que es la mente detras del asesinato

,person_id,transcript
